# ReFRAME library dataset analysis

In [1]:
import os
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

from benchmarking.metrics.base_metrics import load_all_feature_tables
from benchmarking.plots import make_spectrum_plot

tqdm.pandas()

# Load in data

In [2]:
ground_truth_library_data = pd.read_parquet(
    "../data/library_spectra/reframe_spikein_lib.pq"
)

In [3]:
# Get the mzmine based feature tables only
annotation_types = [
    "annotated_spectral_entropy",
    "annotated_cosine_similarity",
    "annotated_dreams_similarity",
]

# Getting the relevant tables
combined_feature_table_list = defaultdict(pd.DataFrame)

for annotation_subfolder in annotation_types:
    combined_feature_table_list[annotation_subfolder] = load_all_feature_tables(
        ["../data/groundtruth_dataset/MSV000098263"],
        annotation_subfolder=annotation_subfolder,
    )

100%|██████████| 1/1 [00:02<00:00,  2.04s/it]


In [4]:
combined_feature_table_list["annotated_spectral_entropy"].keys()

dict_keys(['metaboscape', 'msdial', 'mzmine'])

In [5]:
# Confident annotations are those with a score > 0.7.
confident_annotations = defaultdict(dict)
for ann_type, df in combined_feature_table_list.items():
    for tool, df in df.items():
        confident_annotations[ann_type][tool] = df[df["SCORE"] > 0.7]

In [6]:
# Create a big table of all confident annotations across all tools and annotation types
all_confident_annotations = pd.DataFrame()

for ann_type, tool_dict in confident_annotations.items():
    for tool, df in tool_dict.items():
        df = df.copy()
        df["annotation_type"] = ann_type
        df["tool"] = tool

        # keep only specific columns
        df = df[
            [
                "FEATURE_ID",
                "M/Z",
                "RT",
                "CCS",
                "ADDUCT",
                "MS/MS_MZS",
                "MS/MS_INTENSITIES",
                "SCORE",
                "INCHIKEY",
                "annotation_type",
                "tool",
            ]
        ]

        all_confident_annotations = pd.concat(
            [all_confident_annotations, df], ignore_index=True
        )

all_confident_annotations.head()

,FEATURE_ID,M/Z,RT,CCS,ADDUCT,MS/MS_MZS,MS/MS_INTENSITIES,SCORE,INCHIKEY,annotation_type,tool
0,14,450.24143,0.580667,342.04605,[M+2H]+2,"[44.04925050935498, 55.05411669559103, 56.0496...","[471.01196, 187.40848, 162.61108, 75.35646, 87...",0.725595,PVHLMTREZMEJCG-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape
1,21,367.76988,0.586333,330.99915,[M+2H]+2,"[28.763710695231723, 35.36744190873125, 41.042...","[6.493458, 2.015952, 118.15643, 78.726494, 78....",0.812354,HNDXPZPJZGTJLJ-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape
2,31,528.72389,0.589833,349.22684,[M+2H]+2,"[43.017459271581295, 70.02815680374228, 70.040...","[80.21982, 294.8787, 96.54403, 5241.1816, 1335...",0.903445,BJFIDCADFRDPIO-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape
3,46,614.25617,1.566833,371.47680,[M+2H]+2,"[20.332255497995753, 25.269010136394556, 25.97...","[1.6639208, 3.99059, 0.55967736, 3.99059, 4.16...",0.898144,BENFXAYNYRLAIU-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape
4,106,512.77811,2.322167,353.69818,[M+2H]+2,"[41.03832172082779, 42.0337999275829, 43.01784...","[60.664436, 52.36496, 96.396645, 37.363605, 19...",0.869091,JDKLPDJLXHXHNV-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape


In [7]:
# Map these inchikeys to their names (to seletc only a couple of them for the mirror plots)
import pubchempy as pcp


def inchikey_to_name(inchikey):
    try:
        compound = pcp.get_compounds(inchikey, "inchikey")[0]
        # Get common name if available, otherwise use IUPAC name
        synonyms = compound.synonyms
        return synonyms[0] if synonyms else compound.iupac_name, compound.cid
    except Exception as e:
        print(f"Error retrieving name for {inchikey}: {e}")
        return None, None


def get_annotation_info(df):
    skipped = 0
    found_inchikeys = set()
    for inchikey, group_df in df.groupby("INCHIKEY"):
        num_tools_found = group_df["tool"].nunique()
        num_annotation_types = group_df["annotation_type"].nunique()

        if num_tools_found != 3 and num_annotation_types != 3:
            skipped += 1
            continue

        found_inchikeys.add(inchikey)

    base_df = pd.DataFrame({"INCHIKEY": list(found_inchikeys)})
    base_df["NAME"], base_df["CID"] = zip(
        *base_df["INCHIKEY"].progress_apply(inchikey_to_name)
    )
    return base_df

In [8]:
if not os.path.exists(
    "../data/groundtruth_dataset/MSV000098263/confidently_annotated_inchikey.csv",
):
    base_df = get_annotation_info(all_confident_annotations)
    base_df.to_csv(
        "../data/groundtruth_dataset/MSV000098263/confidently_annotated_inchikey.csv",
        index=False,
    )

# Select the inchikeys to plot

In [9]:
base_df = pd.read_csv(
    "../data/groundtruth_dataset/MSV000098263/confidently_annotated_inchikey.csv"
)
all_confident_annotations = all_confident_annotations.merge(base_df, on="INCHIKEY")
all_confident_annotations.head()

,FEATURE_ID,M/Z,RT,CCS,ADDUCT,MS/MS_MZS,MS/MS_INTENSITIES,SCORE,INCHIKEY,annotation_type,tool,NAME,CID
0,14,450.24143,0.580667,342.04605,[M+2H]+2,"[44.04925050935498, 55.05411669559103, 56.0496...","[471.01196, 187.40848, 162.61108, 75.35646, 87...",0.725595,PVHLMTREZMEJCG-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape,C41H62N12O11,4075169.0
1,21,367.76988,0.586333,330.99915,[M+2H]+2,"[28.763710695231723, 35.36744190873125, 41.042...","[6.493458, 2.015952, 118.15643, 78.726494, 78....",0.812354,HNDXPZPJZGTJLJ-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape,C41H71N3O8,74336136.0
2,31,528.72389,0.589833,349.22684,[M+2H]+2,"[43.017459271581295, 70.02815680374228, 70.040...","[80.21982, 294.8787, 96.54403, 5241.1816, 1335...",0.903445,BJFIDCADFRDPIO-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape,Lys-vasopressin,5772.0
3,46,614.25617,1.566833,371.47680,[M+2H]+2,"[20.332255497995753, 25.269010136394556, 25.97...","[1.6639208, 3.99059, 0.55967736, 3.99059, 4.16...",0.898144,BENFXAYNYRLAIU-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape,DB-042837,9920035.0
4,106,512.77811,2.322167,353.69818,[M+2H]+2,"[41.03832172082779, 42.0337999275829, 43.01784...","[60.664436, 52.36496, 96.396645, 37.363605, 19...",0.869091,JDKLPDJLXHXHNV-UHFFFAOYSA-N,annotated_spectral_entropy,metaboscape,C50H69N15O9,14560360.0


In [10]:
should_be_skipped = set()
for inchikey, group_df in all_confident_annotations.groupby("INCHIKEY"):
    tools_annotated = group_df["tool"].unique()
    annotation_types = group_df["annotation_type"].unique()

    # Check if annotated by the three tools
    if len(tools_annotated) != 3:
        # Check if anntoated by 3 types for each tool
        tool_annotation_counts = group_df.groupby("tool")["annotation_type"].nunique()
        if not all(tool_annotation_counts == 3):
            should_be_skipped.add(inchikey)

subset_confident_annotations = all_confident_annotations[
    ~all_confident_annotations["INCHIKEY"].isin(should_be_skipped)
]
len(subset_confident_annotations), len(all_confident_annotations)

(155831, 158838)

# Plot mirror plots across the three tools and their respective annotation types

In [11]:
inchikeys_of_interest = [
    "ACEWLPOYLGNNHV-UHFFFAOYSA-N",  # Ibuprofen piconol
    "DGBIGWXXNGSACT-UHFFFAOYSA-N",  # Clonazepam
    "GSDSWSVVBLHKDQ-UHFFFAOYSA-N",  # Ofloxacin
    "IQPSEEYGBUAQFF-UHFFFAOYSA-N",  # Pantoprazole
    "ZPUCINDJVBIVPJ-UHFFFAOYSA-N",  # Allococaine
    # "AAOVKJBEBIDNHE-UHFFFAOYSA-N",  # Diazepam
    # "PJVWKTKQMONHTI-UHFFFAOYSA-N",  # Warfarin
    # "XUKUURHRXDUEBC-UHFFFAOYSA-N",  # Atorvastatin(Relative)
    # "ZTVQQQVZCWLTDF-UHFFFAOYSA-N",  # Remifentanil (REMIFENTANIL)
    # "GGCSSNBKKAUURC-UHFFFAOYSA-N",  # Sufentanil (SUFENTANIL)
]

In [12]:
subset_df = subset_confident_annotations[
    subset_confident_annotations["INCHIKEY"].isin(inchikeys_of_interest)
]

for inchikey, group_df in subset_df.groupby("INCHIKEY"):
    group_df.sort_values(
        by=["M/Z", "RT", "SCORE"], ascending=[True, True, False], inplace=True
    )

    df = group_df.copy()

    # Round M/Z and RT to 2 decimal places for grouping
    df["M/Z"] = df["M/Z"].round(2)
    df["RT"] = df["RT"].round(2)

    # Get median m/z and RT for the group
    median_mz = df["M/Z"].median()
    median_rt = df["RT"].median()

    # 5 ppm window around median m/z
    mz_tol = median_mz * 5 / 1e6  # 5 ppm window

    range_mz = (median_mz - mz_tol, median_mz + mz_tol)
    range_rt = (median_rt - 0.1, median_rt + 0.1)  # 0.2 min window

    final_hits = df[
        (df["M/Z"] >= range_mz[0])
        & (df["M/Z"] <= range_mz[1])
        & (df["RT"] >= range_rt[0])
        & (df["RT"] <= range_rt[1])
    ]
    final_hits.sort_values(by="tool", inplace=True)

    # Count the tool-annotation type combinations for the final hits
    tool_ann_combinations = final_hits.groupby(["tool", "annotation_type"]).size()

    # Check you have 3 tools and 3 annotation types for each tool is represented in the final hits
    if tool_ann_combinations.index.get_level_values("tool").nunique() == 3:
        if all(
            tool_ann_combinations.groupby("tool").size() == 3
        ):  # Each tool should have at least 3 annotation types

            # Subset to keep only the top hit for each tool-annotation type combination
            final_hits = final_hits.groupby(["tool", "annotation_type"]).head(1)
            final_hits.sort_values(by=["tool", "annotation_type"], inplace=True)

            annotation_type_mapping = {
                "annotated_spectral_entropy": "Spectral",
                "annotated_cosine_similarity": "Cosine",
                "annotated_dreams_similarity": "DREAMS",
            }

            all_spectra = []
            for idx, row in final_hits.iterrows():
                mzs = row["MS/MS_MZS"]
                ints = row["MS/MS_INTENSITIES"]
                title = f"{annotation_type_mapping[row['annotation_type']]} ({row['SCORE']:.2f})"
                all_spectra.append(
                    {
                        "mzs": mzs,
                        "intensities": ints,
                        "tool": row["tool"],
                        "title": title,
                    }
                )

            inchikey = row["INCHIKEY"]
            name = row["NAME"]

            fig = make_spectrum_plot(
                all_spectra,
                title=f"{inchikey} ({name}) - Confident Annotations Across Tools and Types",
                n_cols=3,
            )

            os.makedirs("../figures/spectrum_plots", exist_ok=True)

            # Save the figure to html
            fig.write_html(f"../figures/spectrum_plots/{inchikey}.html")

            # Save the figure as png
            fig.write_image(f"../figures/spectrum_plots/{inchikey}.png", scale=2)